# 🚀 Face Recognition Testing (FaceNet + SVM)
ไฟล์สำหรับใช้งานและทดสอบระบบจดจำใบหน้าระดับสูง
- โหลดโมเดลที่เทรนจาก `Train_Ai_Pro.ipynb`
- ทดสอบกับโฟลเดอร์ Test
- เปิดกล้อง Live Webcam (Google Colab & Local)


In [ ]:
# 1. เชื่อมต่อ Google Drive
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ เชื่อมต่อ Google Drive สำเร็จ')
    IS_COLAB = True
except ImportError:
    print('ℹ️ รันใน Local Environment')
    IS_COLAB = False


In [ ]:
# 2. ติดตั้งและนำเข้าไลบรารี
import subprocess
import sys

def install_if_missing(package, import_name=None):
    try:
        __import__(import_name or package)
    except ImportError:
        print(f'กำลังติดตั้ง {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

install_if_missing('facenet-pytorch', 'facenet_pytorch')
install_if_missing('scikit-learn', 'sklearn')
install_if_missing('joblib')

import torch
import numpy as np
import cv2
import joblib
import matplotlib.pyplot as plt
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'💻 ใช้หน่วยประมวลผล: {device}')

print('กำลังโหลดโมเดล FaceNet...')
mtcnn = MTCNN(image_size=160, margin=20, keep_all=True, select_largest=False, post_process=False, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
print('✅ โหลดโมเดลสำเร็จ!')


In [ ]:
# 3. โหลดโมเดล SVM และ Label Encoder
possible_model_dirs = [
    '.', '..', '/content/drive/MyDrive/Project_Ai',
    os.path.join(os.getcwd())
]

clf_path = None
le_path = None

for d in possible_model_dirs:
    p1 = os.path.join(d, 'facenet_svm_model.pkl')
    p2 = os.path.join(d, 'label_encoder.pkl')
    if os.path.exists(p1) and os.path.exists(p2):
        clf_path = p1
        le_path = p2
        break

if clf_path and le_path:
    clf = joblib.load(clf_path)
    le = joblib.load(le_path)
    print(f'✅ โหลด SVM Model สำเร็จจาก: {clf_path}')
    print(f'📋 รายชื่อในระบบ: {le.classes_}')
else:
    raise FileNotFoundError('❌ ไม่พบไฟล์ facenet_svm_model.pkl หรือ label_encoder.pkl กรุณารัน Train_Ai_Pro.ipynb ก่อน')


In [ ]:
# 4. ฟังก์ชันหลักสำหรับจดจำใบหน้า
def recognize_faces(img_input, confidence_threshold=0.6, title="Face Recognition"):
    if isinstance(img_input, str):
        if not os.path.exists(img_input):
            print(f'❌ ไม่พบไฟล์ภาพ: {img_input}')
            return
        img = cv2.imread(img_input)
    else:
        img = img_input.copy()

    if img is None:
        print('❌ ไม่สามารถเปิดภาพได้')
        return

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    
    # 1. ค้นหาใบหน้าทั้งหมดในภาพ
    boxes, probs = mtcnn.detect(pil_img)
    display_img = img_rgb.copy()
    
    if boxes is not None:
        # ตัดใบหน้าทีละหน้าเพื่อส่งเข้า ResNet
        faces = mtcnn(pil_img)
        if faces is not None:
            # 2. สกัด Feature (Embeddings)
            faces_tensor = faces.to(device)
            with torch.no_grad():
                embeddings = resnet(faces_tensor).cpu().numpy()
            
            # 3. จำแนกใบหน้าด้วย SVM
            preds = clf.predict(embeddings)
            pred_probs = clf.predict_proba(embeddings)
            
            for i, box in enumerate(boxes):
                prob_max = np.max(pred_probs[i])
                if prob_max >= confidence_threshold:
                    name = le.inverse_transform([preds[i]])[0]
                    color = (0, 255, 0)
                    label = f"{name} {prob_max*100:.1f}%"
                else:
                    name = "Unknown"
                    color = (255, 0, 0)
                    label = f"{name} {prob_max*100:.1f}%"
                
                # วาดกรอบและชื่อ
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(display_img, (x1, y1), (x2, y2), color, 3)
                
                # พื้นหลังข้อความ
                bw = int(len(label) * 15)
                cv2.rectangle(display_img, (x1, max(0, y1-30)), (x1+bw, max(0, y1)), color, -1)
                cv2.putText(display_img, label, (x1+5, max(15, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
        print(f'🔍 ตรวจพบ {len(boxes)} ใบหน้า')
    else:
        print('🔍 ไม่พบใบหน้าในภาพนี้')

    plt.figure(figsize=(10, 8))
    plt.imshow(display_img)
    plt.title(title)
    plt.axis('off')
    plt.show()

# ทดสอบกับภาพในโฟลเดอร์ test (ถ้ามี)
possible_test_dirs = ['test', './test', '../test', '/content/drive/MyDrive/Project_Ai/test']
for tf in possible_test_dirs:
    if os.path.exists(tf) and len(os.listdir(tf)) > 0:
        sample = sorted(os.listdir(tf))[0]
        recognize_faces(os.path.join(tf, sample), title="Image Test")
        break


In [ ]:
# 5. Live Webcam (Google Colab)
from IPython.display import display, Javascript
from base64 import b64decode

def run_webcam_colab():
    js = Javascript("""
        async function takePhoto() {
            const div = document.createElement('div');
            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.maxWidth = '480px';
            div.appendChild(video);
            
            const capture = document.createElement('button');
            capture.textContent = '📸 ถ่ายภาพ & วิเคราะห์';
            capture.style.padding = '10px 20px';
            capture.style.marginTop = '10px';
            div.appendChild(capture);
            
            document.body.appendChild(div);
            
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();
            
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            await new Promise((resolve) => capture.onclick = resolve);
            
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', 0.9);
        }
    """)
    display(js)
    try:
        from google.colab.output import eval_js
        print("กำลังเปิดกล้อง... (กดปุ่ม 'ถ่ายภาพ' เพื่อจับภาพและวิเคราะห์)")
        data = eval_js('takePhoto()')
        binary = b64decode(data.split(',')[1])
        img_arr = np.frombuffer(binary, dtype=np.uint8)
        img_cv = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        
        print("✅ ถ่ายภาพสำเร็จ! กำลังประมวลผล...")
        recognize_faces(img_cv, title="Live Webcam Test")
    except Exception as e:
        print(f"⚠️ ไม่สามารถเปิดกล้องได้: {e}")

# วิธีใช้งาน:
# if IS_COLAB:
#     run_webcam_colab()


In [ ]:
# 6. Live Webcam (Local PC - สำหรับรันในเครื่องตัวเอง)
def test_local_webcam():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ ไม่สามารถเปิดกล้องได้")
        return

    print("🎥 เปิดกล้องสำเร็จ: กด [SPACE] ถ่ายภาพและวิเคราะห์ | กด [Q] ปิดกล้อง")

    while True:
        ret, frame = cap.read()
        if not ret: break

        cv2.putText(frame, "SPACE=Capture | Q=Quit", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        cv2.imshow("Live Webcam", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord(' '):
            captured_frame = frame.copy()
            cap.release()
            cv2.destroyAllWindows()
            print("📸 ถ่ายภาพแล้ว กำลังประมวลผล...")
            recognize_faces(captured_frame, title="Local Webcam")
            break
        elif key == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            print("ปิดกล้องเรียบร้อย")
            break

# วิธีใช้งาน:
# if not IS_COLAB:
#     test_local_webcam()
